In [1]:
"""
ZeptoFresh EDA & Feature Engineering Script
Author: Your Name
Description:
- Generates synthetic dataset
- Performs data cleaning
- Conducts EDA
- Creates engineered features
"""

import pandas as pd
import numpy as np

# -----------------------------
# 1. Synthetic Data Generation
# -----------------------------
def generate_synthetic_data(n=10000):
    np.random.seed(42)

    df = pd.DataFrame({
        "order_id": range(1, n+1),
        "hub_id": np.random.randint(1, 50, n),
        "city": np.random.choice(["Mumbai", "Bangalore", "Delhi", "Pune", "Ahmedabad"], n),
        "order_category": np.random.choice(["Grocery", "Fresh Food", "Bakery", "Medicines"], n),
        "order_value_Rs": np.random.exponential(scale=500, size=n),
        "items_count": np.random.randint(1, 10, n),
        "delivery_time_mins": np.random.normal(18, 8, n),
        "prep_time_mins": np.random.normal(10, 5, n),
        "rider_distance_km": np.random.uniform(0.5, 10, n),
        "order_hour": np.random.randint(0, 24, n),
        "is_weekend": np.random.choice([0, 1], n),
        "rain_flag": np.random.choice([0, 1], n),
        "customer_age": np.random.randint(18, 60, n),
        "customer_tenure_days": np.random.randint(1, 1000, n),
        "coupon_used": np.random.choice([0, 1], n),
        "tip_amount_Rs": np.random.exponential(scale=30, size=n),
        "refund_issued": np.random.choice([0, 1], n),
        "customer_rating": np.random.choice([1, 2, 3, 4, 5, 0, np.nan], n)
    })

    # Introduce anomalies
    df.loc[np.random.choice(df.index, 50), "delivery_time_mins"] = 0
    df.loc[np.random.choice(df.index, 30), "prep_time_mins"] = -5
    df.loc[np.random.choice(df.index, 1), "order_value_Rs"] = 295000

    return df


# -----------------------------
# 2. Data Cleaning
# -----------------------------
def clean_data(df):
    df = df.copy()

    # Remove invalid delivery times
    df = df[df["delivery_time_mins"] > 0]

    # Fix negative prep time
    df["prep_time_mins"] = df["prep_time_mins"].apply(lambda x: max(x, 0))

    # Handle customer_rating
    df["customer_rating"] = df["customer_rating"].replace(0, np.nan)
    df["customer_rating"].fillna(df["customer_rating"].median(), inplace=True)

    # Cap extreme order values
    upper_limit = df["order_value_Rs"].quantile(0.99)
    df["order_value_Rs"] = np.where(df["order_value_Rs"] > upper_limit,
                                   upper_limit,
                                   df["order_value_Rs"])

    return df


# -----------------------------
# 3. Exploratory Data Analysis
# -----------------------------
def perform_eda(df):
    print("\n--- BASIC STATS ---")
    print(df.describe())

    print("\n--- DELIVERY TIME DISTRIBUTION ---")
    print("Mean:", df["delivery_time_mins"].mean())
    print("Median:", df["delivery_time_mins"].median())

    print("\n--- CORRELATION MATRIX ---")
    print(df.corr(numeric_only=True)[["delivery_time_mins"]].sort_values(by="delivery_time_mins", ascending=False))

    print("\n--- GROUPED ANALYSIS (CITY) ---")
    print(df.groupby("city")["delivery_time_mins"].mean())


# -----------------------------
# 4. Feature Engineering
# -----------------------------
def create_features(df):
    df = df.copy()

    df["delivery_speed"] = df["rider_distance_km"] / df["delivery_time_mins"]

    df["delay_ratio"] = df["delivery_time_mins"] / (df["prep_time_mins"] + 1)

    df["high_value_order"] = (df["order_value_Rs"] > 1000).astype(int)

    df["is_peak_hour"] = df["order_hour"].apply(lambda x: 1 if 18 <= x <= 22 else 0)

    return df


# -----------------------------
# 5. Main Pipeline
# -----------------------------
def main():
    print("Generating synthetic dataset...")
    df = generate_synthetic_data(10000)

    print("\nCleaning data...")
    df_clean = clean_data(df)

    print("\nPerforming EDA...")
    perform_eda(df_clean)

    print("\nCreating features...")
    df_final = create_features(df_clean)

    print("\nFinal dataset shape:", df_final.shape)
    print(df_final.head())


# -----------------------------
# Run Script
# -----------------------------
if __name__ == "__main__":
    main()

Generating synthetic dataset...

Cleaning data...

Performing EDA...

--- BASIC STATS ---
           order_id       hub_id  order_value_Rs  items_count  \
count   9851.000000  9851.000000     9851.000000  9851.000000   
mean    4996.567760    25.150036      500.675437     5.028017   
std     2882.591597    14.081547      478.404507     2.564345   
min        1.000000     1.000000        0.010259     1.000000   
25%     2502.500000    13.000000      148.358314     3.000000   
50%     4995.000000    25.000000      350.667107     5.000000   
75%     7488.500000    37.000000      703.788728     7.000000   
max    10000.000000    49.000000     2307.500240     9.000000   

       delivery_time_mins  prep_time_mins  rider_distance_km   order_hour  \
count         9851.000000     9851.000000        9851.000000  9851.000000   
mean            18.203962       10.024762           5.264371    11.508273   
std              7.741063        4.886838           2.720328     6.903498   
min             

C:\Users\Zenil\AppData\Local\Temp\ipykernel_20264\1838499468.py:63: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["customer_rating"].fillna(df["customer_rating"].median(), inplace=True)
